
# NBA SDP Pipeline – Data Inspection & Validation

This notebook validates the SQL-based NBA Medallion Architecture implemented using Databricks Spark Declarative Pipelines.

It covers:
- Inspection of raw JSON data and schema
- Record counts across Bronze, Silver, and Gold
- Data quality and game ID uniqueness
- Investigation of invalid or incomplete scoring records
- Validation of Gold aggregations and derived metrics

**Note:** The notebook requires the NBA source file and the corresponding Unity Catalog tables to exist in the Databricks workspace.


In [0]:

file_path = "/Volumes/nba/default/nba_data/alle_spiele.json"

# Show first 1200 characters of the JSON file
print(dbutils.fs.head(file_path, 1200))


In [0]:
df = (
    spark.read
    .option("multiLine","true")
    .json(file_path)
)

df.printSchema()

In [0]:
%sql

-- 01 Row counts across each Medallion layer

SELECT 'Bronze' AS layer, COUNT(*) AS records
FROM nba.sdp_training.bronze_games_sdp_training

UNION ALL

SELECT 'Silver', COUNT(*)
FROM nba.sdp_training.silver_games_sdp_training

UNION ALL

SELECT 'Gold', COUNT(*)
FROM nba.sdp_training.gold_team_scoring_sdp;


layer,records
Bronze,125
Silver,59
Gold,11


In [0]:
%sql

-- 02 Silver data quality and uniqueness

SELECT
    COUNT(*) AS silver_games,
    COUNT(DISTINCT id) AS unique_game_ids,
    MIN(total_points) AS min_total_points,
    MAX(total_points) AS max_total_points
FROM nba.sdp_training.silver_games_sdp_training;


In [0]:
%sql

-- 03 Investigate invalid source scores

SELECT
    COUNT(*) AS total_games,
    SUM(
        CASE
            WHEN home_team_score IS NULL
              OR visitor_team_score IS NULL
              OR home_team_score <= 0
              OR visitor_team_score <= 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_scores
FROM nba.sdp_training.bronze_games_sdp_training;


In [0]:
%sql

-- 04 Inspect low-scoring records

SELECT
    id,
    date,
    status,
    home_team_name,
    visitor_team_name,
    home_team_score,
    visitor_team_score,
    total_points
FROM nba.sdp_training.silver_games_sdp_training
ORDER BY total_points ASC
LIMIT 10;


In [0]:
%sql

-- 05 Validate Gold aggregation

SELECT
    home_team_name,
    total_games,
    avg_total_points,
    max_total_points,
    high_scoring_games
FROM nba.sdp_training.gold_team_scoring_sdp
ORDER BY total_games DESC, home_team_name;


In [0]:
%sql

-- 06 Validate total points calculation

SELECT
    COUNT(*) AS incorrect_total_points
FROM nba.sdp_training.silver_games_sdp_training
WHERE total_points <> home_team_score + visitor_team_score
   OR total_points IS NULL;
